# Exercise 6: Basic Image Generation

**CAS Agentic AI & Deep Learning - Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Implement and train a **Generative Adversarial Network (GAN)**
- Implement the **reparameterization trick** and train a **Variational Autoencoder (VAE)**
- Train a **Diffusion Model (DM)** using `diffusers` components
- Implement **Flow Matching** and compare its sampling speed with diffusion
- *(Bonus)* Turn images into tokens with a **VQ-VAE** and generate them **autoregressively** with a Transformer
- Compare all models in terms of sample quality, training and sampling cost
- Visualize and interpret samples generated by each model type

**Fill in code where you see `# YOUR CODE HERE`.**

<div style="background-color:#fff3cd;border-left:6px solid #ff9900;padding:14px 18px;margin:12px 0;border-radius:4px;color:#222">
<div style="font-size:18px;font-weight:bold">⚡ This exercise needs a GPU runtime</div>
<div style="margin-top:8px">
<b>In Google Colab:</b> click <code>Runtime → Change runtime type → T4 GPU</code> (free tier).<br>
Training generative models on CPU works but takes 10–20 minutes per section.
</div>
</div>

## Setup Colab / Local Paths

The following setup sets the default data path.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")

if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    %pip install -q diffusers accelerate lightning torchinfo
    # Colab's preinstalled torchao is too old for diffusers and breaks `import diffusers`;
    # we do not use it, so remove it
    %pip uninstall -q -y torchao

    ROOT_PATH = Path("/content")
    DATA_PATH = ROOT_PATH.joinpath("data")

else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

If you want to mount your Google Drive you can use the following function. This will enable you to persist data and model checkpoints across sessions.

In [ ]:
def mount_drive(
    drive_mount_path: str = "/content/drive", data_path: str = "MyDrive/cas-dl-compvis"
) -> tuple[Path, Path]:
    from google.colab import drive

    root_path = Path(drive_mount_path)
    drive.mount(str(root_path))
    data_path = root_path.joinpath(data_path)
    return root_path, data_path


# Uncomment to mount Google Drive (persists data between sessions):
# ROOT_PATH, DATA_PATH = mount_drive()

## Helper Functions

In [ ]:
import math
import time
from textwrap import wrap

import numpy as np
import seaborn as sns
import torch
from matplotlib import pyplot as plt
from torchvision.transforms.v2 import functional as TF
from torchvision.utils import make_grid


def plot_collage(
    images,
    captions=None,
    nrows=None,
    ncols=None,
    caption_width=30,
    caption_font_size=10,
    figsize=None,
    title=None,
):
    """Plot a list of PIL images or [0, 1] tensors in a grid, optionally with captions."""
    num_images = len(images)
    if nrows is None or ncols is None:
        nrows = ncols = math.ceil(math.sqrt(num_images))
    if figsize is None:
        figsize = (ncols * 2, nrows * 2)
    fig, axes = plt.subplots(figsize=figsize, nrows=nrows, ncols=ncols, squeeze=False)
    if title:
        fig.suptitle(title, fontsize=20)
    for i, ax in enumerate(axes.flatten()):
        ax.set_xticks([])
        ax.set_yticks([])
        if i >= num_images:
            ax.axis("off")
            continue
        img = images[i]
        if isinstance(img, torch.Tensor):
            img = TF.to_pil_image(img)
        _ = ax.imshow(np.array(img), cmap="gray", vmin=0, vmax=255)
        if captions is not None and i < len(captions):
            caption = "\n".join(wrap(captions[i], caption_width))
            _ = ax.set_title(caption, fontsize=caption_font_size)
    return fig, axes


def show_samples(samples, nrow=4, title=None):
    """Show a batch of [0, 1] image tensors (N, C, H, W) as a single grid."""
    grid = make_grid(samples.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    ncols = min(nrow, len(samples))
    nrows = math.ceil(len(samples) / nrow)
    fig, ax = plt.subplots(figsize=(ncols * 1.0, nrows * 1.0))
    _ = ax.imshow(grid.permute(1, 2, 0).numpy())
    _ = ax.axis("off")
    if title:
        _ = ax.set_title(title)
    return fig, ax


def denorm(x):
    """Map images from [-1, 1] back to [0, 1] for display."""
    return (x * 0.5 + 0.5).clamp(0, 1)

## Imports

In [ ]:
from collections.abc import Callable

import lightning as L
from diffusers import DDPMScheduler, UNet2DModel
from lightning.pytorch.loggers import TensorBoardLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary
from torchvision import transforms
from torchvision.datasets import MNIST

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device} | Colab: {IN_COLAB}")

if IN_COLAB and device.type == "cpu":
    from IPython.display import HTML, display

    display(
        HTML("""
    <div style="background:#ffebee;border:3px solid #d32f2f;border-radius:8px;
                padding:16px;margin:12px 0;font-size:16px;color:#000">
        <div style="font-size:20px;font-weight:bold;color:#d32f2f">
            ⚠️ NO GPU DETECTED — training will be very slow
        </div>
        <div style="margin-top:8px">
            Switch to a GPU runtime: <b>Runtime → Change runtime type → T4 GPU</b>
            (free tier), then <b>Runtime → Run all</b>.
        </div>
    </div>
    """)
    )

## Configuration and Shared Utilities

Reduce the number of epochs below if training is too slow.

Each model section below (GAN, VAE, DDPM, FM, bonus AR) only depends on the cells up to and including **0) Prepare Dataset**, so you can skip sections. Every trained model is added to `TRAINED_MODELS` (with its training time in `TIMINGS`).

In [ ]:
BATCH_SIZE = 128
NUM_WORKERS = 2
SEED = 123
LOG_PATH = DATA_PATH.joinpath("lightning_logs")

GAN_LATENT_DIM = 20
GAN_EPOCHS = 10

VAE_LATENT_DIM = 2
VAE_EPOCHS = 10

DDPM_T = 1000  # number of diffusion steps
DDPM_EPOCHS = 5  # increase to 15+ for better quality
DDPM_MONITOR_EVERY_N_EPOCHS = 5  # DDPM sampling (1000 steps) is slow

FM_EPOCHS = 5  # same budget as DDPM for a fair comparison
FM_SAMPLE_STEPS = 20

# Mixed precision makes the UNet (DDPM, FM) faster on GPU
UNET_PRECISION = "16-mixed" if torch.cuda.is_available() else "32-true"

# Bonus: autoregressive model
VQ_NUM_CODES = 64
VQ_CODE_DIM = 16
VQ_EPOCHS = 5
AR_EPOCHS = 10

TRAINED_MODELS = {}
TIMINGS = {}

In [ ]:
TO_TENSOR = transforms.ToTensor()  # [0, 1]
TO_PM1 = transforms.Compose([TO_TENSOR, transforms.Normalize(0.5, 0.5)])  # [-1, 1]
TO_PM1_32 = transforms.Compose([TO_PM1, transforms.Resize(32)])  # [-1, 1], 32x32


class SampleGridMonitor(L.Callback):
    """Log the images returned by `sample_fn(pl_module)` (in [0, 1]) to TensorBoard."""

    def __init__(self, sample_fn, tag="samples", nrow=4, every_n_epochs=1):
        super().__init__()
        self.sample_fn = sample_fn
        self.tag = tag
        self.nrow = nrow
        self.every_n_epochs = every_n_epochs

    def on_train_epoch_end(self, trainer, pl_module):
        if (trainer.current_epoch + 1) % self.every_n_epochs != 0:
            return
        pl_module.eval()
        with torch.no_grad():
            images = self.sample_fn(pl_module)
        pl_module.train()
        grid = make_grid(images, nrow=self.nrow)
        pl_module.logger.experiment.add_image(self.tag, grid, trainer.current_epoch)


def fit_model(name, model, datamodule, max_epochs, callbacks=(), precision="32-true"):
    """Train `model`, then register it and its training time under `name`."""
    trainer = L.Trainer(
        accelerator="auto",
        devices=1,
        max_epochs=max_epochs,
        precision=precision,
        logger=TensorBoardLogger(LOG_PATH, name=name.lower()),
        default_root_dir=LOG_PATH,
        callbacks=list(callbacks),
    )
    start = time.perf_counter()
    trainer.fit(model, datamodule)
    train_s = time.perf_counter() - start
    TRAINED_MODELS[name] = model
    TIMINGS[name] = {"train_s": train_s, "epochs": max_epochs}
    print(f"{name}: trained {max_epochs} epochs in {train_s:.0f}s")


def make_unet():
    """Small UNet for 32x32 grayscale images (resolutions 32 -> 16 -> 8, attention at 8x8)."""
    return UNet2DModel(
        sample_size=32,
        in_channels=1,
        out_channels=1,
        layers_per_block=1,
        block_out_channels=(32, 64, 128),
        down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D"),
        up_block_types=("AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    )

## 0) Prepare Dataset

We use the **MNIST** dataset throughout this exercise. Its small image size (28x28 grayscale) and clean structure make it ideal for comparing generative model architectures within a reasonable training time.

In [ ]:
ds_mnist_train = MNIST(root=DATA_PATH.joinpath("mnist"), train=True, download=True)
print(f"Training samples: {len(ds_mnist_train)}")

Let's inspect a few samples from the dataset.

In [ ]:
images = [ds_mnist_train[i][0] for i in range(16)]
labels = [f"Label: {ds_mnist_train[i][1]}" for i in range(16)]
fig, ax = plot_collage(images, captions=labels)

We create a Lightning `DataModule` to handle data loading consistently across all three models.

In [ ]:
class MNISTDataModule(L.LightningDataModule):
    def __init__(
        self,
        data_dir: str,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        transform_fn: Callable | None = None,
    ):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.transform = transform_fn or TO_TENSOR

    def prepare_data(self):
        MNIST(self.data_dir, train=True, download=True)

    def setup(self, stage=None):
        self.ds_train = MNIST(self.data_dir, train=True, transform=self.transform)

    def train_dataloader(self):
        return DataLoader(
            self.ds_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.num_workers > 0,
        )

In [ ]:
data_module = MNISTDataModule(data_dir=DATA_PATH.joinpath("mnist"))
data_module.setup("fit")
image, label = data_module.ds_train[0]
print(f"Image shape: {image.shape}, dtype: {image.dtype}, label: {label}")

## 1) Generative Adversarial Networks (GAN)

*This section only needs the cells up to and including **0) Prepare Dataset**. You can skip the other model sections.*

A GAN consists of two networks trained in an adversarial manner.
- **Generator** $\mathcal{G}$: maps a random latent vector $z \sim \mathcal{N}(0, I)$ to a synthetic image.
- **Discriminator** $\mathcal{D}$: predicts whether an image is real or generated.

Training alternates between:
1. **Generator step**: $\mathcal{G}$ tries to fool $\mathcal{D}$, minimise $\mathcal{D}$'s confidence on fake images.
2. **Discriminator step**: $\mathcal{D}$ tries to distinguish real from fake, minimise classification error on both.

You will implement the two **loss functions**; the training loop around them is given.

The minimax objective is:
$$\min_G \max_D \; \mathbb{E}_{x}[\log D(x)] + \mathbb{E}_z[\log(1 - D(G(z)))]$$

### Generator

Maps a latent vector to a 28x28 image via transposed convolutions.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=GAN_LATENT_DIM):
        super().__init__()
        self.latent_dim = latent_dim
        self.model = nn.Sequential(
            nn.Linear(self.latent_dim, 6 * 49),
            nn.Unflatten(1, (6, 7, 7)),
            nn.ConvTranspose2d(
                6, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.LeakyReLU(),
            nn.BatchNorm2d(32),  # BN after the activation: BN -> act collapses here
            nn.ConvTranspose2d(
                32, 64, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.LeakyReLU(),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 32, kernel_size=3, stride=1, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 1, kernel_size=1, stride=1, padding=0),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, z):
        return self.model(z)


generator = Generator(latent_dim=GAN_LATENT_DIM)
summary(generator, input_size=(32, GAN_LATENT_DIM))

### Discriminator

Classifies a 28x28 image as real (1) or fake (0) via convolutions.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.LeakyReLU(),
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 1),
        )

    def forward(self, img):
        return self.model(img)


discriminator = Discriminator()
summary(discriminator, input_size=(32, 1, 28, 28))

### Task 1.1: Generator Loss

The generator wants the discriminator to classify its fakes as **real**. Implement `generator_loss(logits_fake)`: the binary cross-entropy between the discriminator's logits on fake images and a target of `1` for every image.

*Hint*: [F.binary_cross_entropy_with_logits(logits, targets)](https://docs.pytorch.org/docs/2.14/generated/torch.nn.functional.binary_cross_entropy_with_logits.html) takes raw logits (no sigmoid needed).

In [ ]:
# YOUR CODE HERE
def generator_loss(logits_fake: torch.Tensor) -> torch.Tensor:
    """Non-saturating generator loss: fakes should be classified as real (1)."""
    return F.binary_cross_entropy_with_logits(logits_fake, torch.ones_like(logits_fake))

In [ ]:
# A discriminator that is unsure (logit 0 -> p=0.5) gives a loss of -ln(0.5)
assert torch.isclose(generator_loss(torch.zeros(8, 1)), torch.tensor(-math.log(0.5)))
# Fakes that fully fool the discriminator give a loss close to 0
assert generator_loss(torch.full((8, 1), 10.0)) < 1e-3
print("OK")

### Task 1.2: Discriminator Loss

The discriminator is a binary classifier: real images → `1`, fake images → `0`.

Implement `discriminator_loss(logits_real, logits_fake, smoothing)`:
1. BCE between `logits_real` and a target of `1 - smoothing`.
2. BCE between `logits_fake` and a target of `0`.
3. Return the average of the two.

The `smoothing` trick is called **one-sided label smoothing**: the discriminator is never asked to be 100% sure that an image is real, which keeps it from becoming over-confident and gives the generator more useful gradients.

In [ ]:
# YOUR CODE HERE
def discriminator_loss(
    logits_real: torch.Tensor, logits_fake: torch.Tensor, smoothing: float = 0.1
) -> torch.Tensor:
    """BCE on real (target 1 - smoothing) and fake (target 0) images, averaged."""
    targets_real = torch.full_like(logits_real, 1.0 - smoothing)
    loss_real = F.binary_cross_entropy_with_logits(logits_real, targets_real)
    loss_fake = F.binary_cross_entropy_with_logits(
        logits_fake, torch.zeros_like(logits_fake)
    )
    return 0.5 * (loss_real + loss_fake)

In [ ]:
zeros = torch.zeros(8, 1)
confident_real, confident_fake = torch.full((8, 1), 10.0), torch.full((8, 1), -10.0)

# An unsure discriminator (p=0.5 everywhere) gives -ln(0.5)
assert torch.isclose(discriminator_loss(zeros, zeros), torch.tensor(-math.log(0.5)))
# A perfect, confident discriminator has ~0 loss without smoothing ...
assert discriminator_loss(confident_real, confident_fake, smoothing=0.0) < 1e-3
# ... but is penalised for over-confidence with smoothing
assert discriminator_loss(confident_real, confident_fake, smoothing=0.1) > 0.4
print("OK")

### GAN Lightning Module

Read through the training step: it uses your two losses. Because two networks with two optimizers are trained in alternation, it uses Lightning's **manual optimization** (`automatic_optimization = False`): we call `backward` and `optimizer.step()` ourselves for each sub-network.

Note the `x_fake.detach()` in the discriminator step: the discriminator update must not send gradients into the generator.

In [ ]:
class GAN(L.LightningModule):
    def __init__(self, latent_dim=GAN_LATENT_DIM, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.generator = Generator(latent_dim=latent_dim)
        self.discriminator = Discriminator()
        self.latent_dim = latent_dim
        self.automatic_optimization = False  # we manage both optimizers manually

    def forward(self, z):
        return self.generator(z)

    def training_step(self, batch, batch_idx):
        x_real, _ = batch
        optimizer_g, optimizer_d = self.optimizers()
        z = torch.randn((x_real.shape[0], self.latent_dim), device=self.device)

        # ==== Generator step: try to fool the discriminator ====
        self.toggle_optimizer(optimizer_g)
        x_fake = self.generator(z)
        g_loss = generator_loss(self.discriminator(x_fake))
        self.manual_backward(g_loss)
        optimizer_g.step()
        optimizer_g.zero_grad()
        self.untoggle_optimizer(optimizer_g)

        # ==== Discriminator step: tell real from fake ====
        self.toggle_optimizer(optimizer_d)
        d_loss = discriminator_loss(
            self.discriminator(x_real), self.discriminator(x_fake.detach())
        )
        self.manual_backward(d_loss)
        optimizer_d.step()
        optimizer_d.zero_grad()
        self.untoggle_optimizer(optimizer_d)

        self.log_dict({"g_loss": g_loss, "d_loss": d_loss}, prog_bar=True)

    def configure_optimizers(self):
        return (
            torch.optim.Adam(self.generator.parameters(), lr=self.hparams.lr),
            torch.optim.Adam(self.discriminator.parameters(), lr=self.hparams.lr),
        )

**Question (before training)**: You will see `g_loss` and `d_loss` in the progress bar and in TensorBoard. If `g_loss` steadily decreases, does that mean the generated images are getting better?

<details>
<summary>Click to reveal answer</summary>

**Not necessarily**. GAN losses are not a quality measure. Each loss is measured against an opponent that keeps changing. A falling `g_loss` can mean the generator improved, *or* that the discriminator got worse. In practice you judge GAN training by **looking at samples** (that is what the TensorBoard image tab is for) or with sample-based metrics such as FID.

</details>

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir={LOG_PATH} --host 0.0.0.0 --port=6006

In [ ]:
_ = L.seed_everything(SEED)

# Normalize to [-1, 1] to match the Tanh output of the generator
data_module_gan = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_PM1
)
gan_model = GAN(latent_dim=GAN_LATENT_DIM)


def gan_monitor_samples(model, n=16):
    z = torch.randn(n, model.latent_dim, device=model.device)
    return denorm(model(z))


fit_model(
    "GAN",
    gan_model,
    data_module_gan,
    max_epochs=GAN_EPOCHS,
    callbacks=[SampleGridMonitor(gan_monitor_samples, tag="gan/samples")],
)

**Question**: What loss value do you expect for a well-trained GAN where both networks are equally strong? Compare with the values in the progress bar.

<details>
<summary>Click to reveal answer</summary>

At the theoretical equilibrium the generator's samples are indistinguishable from real data, so the best the discriminator can do is guess: $D(x) = 0.5$ for every image. A logit of 0 gives a BCE of $\ln 0.5 \approx 0.69$. **both `g_loss` and `d_loss` idellay hover around 0.69**.

In practice the losses oscillate around (not exactly at) these values. If `d_loss` drops towards 0, the discriminator is winning and the generator receives little useful signal.

</details>

### Task 1.3: Sample from the GAN

Sample 64 latent vectors $z \sim \mathcal{N}(0, I)$, generate images with the trained `gan_model` and display them with `show_samples(..., nrow=8)`. Remember that the generator outputs values in $[-1, 1]$ (use `denorm`).

In [ ]:
# YOUR CODE HERE
_ = gan_model.eval()
z = torch.randn(64, GAN_LATENT_DIM, device=gan_model.device)
with torch.no_grad():
    samples = gan_model(z)

fig, ax = show_samples(denorm(samples), nrow=8, title="GAN samples")
plt.show()

**Question**: Look at the 64 samples. Do all ten digits appear roughly equally often? Do some samples look almost identical?

<details>
<summary>Click to reveal answer</summary>

Some digits may be over-represented, while others are rare or missing. This is (partial) **mode collapse**: the generator learns to produce only a limited variety of images instead of covering the full data distribution. Although the ideal GAN would reproduce all digits in their correct proportions, training can get stuck because the generator receives feedback only on images it already produces, making it difficult to discover missing digits.

VAEs and diffusion models train directly on all training images, providing a learning signal for every digit and generally encouraging better coverage.
</details>

## 2) Variational Autoencoders (VAE)

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

A VAE learns a probabilistic latent space by optimising the **Evidence Lower Bound (ELBO)**:

$$\mathcal{L}_{\text{ELBO}} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{reconstruction}} - \underbrace{D_{\text{KL}}(q(z|x) \| p(z))}_{\text{regularisation}}$$

The encoder maps $x$ to a distribution $q(z|x) = \mathcal{N}(\mu, \sigma^2)$; the decoder maps samples $z$ back to the image space.

The key trick enabling backpropagation through the sampling step is the **reparameterisation trick**:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

### Task 2.1: Reparameterisation Trick

Implement `reparameterize(z_mean, z_log_var)`. Given the mean and log-variance of the latent distribution, return a sample using the reparameterisation trick.

*Hint*: `std = exp(0.5 * z_log_var)`, then sample `eps ~ N(0,I)` using [torch.randn_like(std)](https://docs.pytorch.org/docs/2.14/generated/torch.randn_like.html).

In [ ]:
# YOUR CODE HERE
def reparameterize(z_mean: torch.Tensor, z_log_var: torch.Tensor) -> torch.Tensor:
    """Sample from q(z|x) = N(z_mean, exp(z_log_var)) using the reparameterisation trick."""
    std = torch.exp(0.5 * z_log_var)
    eps = torch.randn_like(std)
    return z_mean + eps * std

In [ ]:
z_mean_test = torch.zeros(10_000, 2)
z_log_var_test = torch.full((10_000, 2), math.log(4.0))  # variance 4 -> std 2
z_sample = reparameterize(z_mean_test, z_log_var_test)
assert z_sample.shape == z_mean_test.shape, f"Got {z_sample.shape}"
assert torch.allclose(z_sample.std(dim=0), torch.tensor(2.0), atol=0.1), (
    "The samples should have std = exp(0.5 * log_var)"
)
# Gradients must flow back to the mean (that is the point of the trick)
z_mean_grad = torch.zeros(4, 2, requires_grad=True)
reparameterize(z_mean_grad, torch.zeros(4, 2)).sum().backward()
assert z_mean_grad.grad is not None
print("OK")

### Encoder and Decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(7 * 7 * 64, 16),
            nn.ReLU(),
        )
        self.z_mean = nn.Linear(16, latent_dim)
        self.z_log_var = nn.Linear(16, latent_dim)

    def forward(self, x):
        pre_z = self.network(x)
        z_mean = self.z_mean(pre_z)
        z_log_var = self.z_log_var(pre_z)
        z = reparameterize(z_mean, z_log_var)
        return z_mean, z_log_var, z


class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(latent_dim, 7 * 7 * 64), nn.ReLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(
                64, 64, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.ReLU(),
            nn.ConvTranspose2d(
                64, 32, kernel_size=3, stride=2, padding=1, output_padding=1
            ),
            nn.ReLU(),
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),  # output in [0, 1]
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(-1, 64, 7, 7)
        return self.deconv(x)


encoder = Encoder(latent_dim=VAE_LATENT_DIM)
summary(encoder, input_size=(16, 1, 28, 28))

### Task 2.2: VAE Loss

Implement `vae_loss(x, x_rec, z_mean, z_log_var)`, the negative ELBO. Return the tuple `(loss, rec_loss, kl_loss)`:

1. **Reconstruction loss**: binary cross-entropy per pixel between `x_rec` and `x`, **summed** over the pixel dimensions and **averaged** over the batch.
2. **KL divergence** to the prior $\mathcal{N}(0, I)$, summed over latent dimensions and averaged over the batch:
   $\mathcal{L}_{KL} = -\frac{1}{2} \sum_j (1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)$
3. `loss = rec_loss + kl_loss`

*Hint*: `F.binary_cross_entropy(x_rec, x, reduction="none")` keeps the per-pixel values.

In [ ]:
# YOUR CODE HERE
def vae_loss(x, x_rec, z_mean, z_log_var):
    """Negative ELBO = reconstruction loss + KL divergence (both averaged over the batch)."""
    rec_loss = (
        F.binary_cross_entropy(x_rec, x, reduction="none").sum(dim=(1, 2, 3)).mean()
    )
    kl_loss = (
        -0.5 * torch.sum(1 + z_log_var - z_mean**2 - torch.exp(z_log_var), dim=1).mean()
    )
    return rec_loss + kl_loss, rec_loss, kl_loss

In [ ]:
x_test = torch.randint(0, 2, (4, 1, 28, 28)).float()  # binary images
mu0, logvar0 = torch.zeros(4, 2), torch.zeros(4, 2)

# A perfect reconstruction with a posterior equal to the prior has ~0 loss
loss, rec_loss, kl_loss = vae_loss(x_test, x_test, mu0, logvar0)
assert rec_loss < 1e-3 and kl_loss.abs() < 1e-6, (rec_loss, kl_loss)
# Shifting the mean by 1 in each of the 2 latent dims costs KL = 2 * 0.5 = 1 per image
_, _, kl_loss = vae_loss(x_test, x_test, torch.ones(4, 2), logvar0)
assert torch.isclose(kl_loss, torch.tensor(1.0)), kl_loss
# A useless reconstruction (0.5 everywhere) costs 784 * ln(2) per image
_, rec_loss, _ = vae_loss(x_test, torch.full_like(x_test, 0.5), mu0, logvar0)
assert torch.isclose(rec_loss, torch.tensor(784 * math.log(2))), rec_loss
print("OK")

### VAE Lightning Module

The encoder already samples $z$ with your `reparameterize`; the module plugs everything into `vae_loss`.

In [ ]:
class VAE(L.LightningModule):
    def __init__(self, latent_dim=VAE_LATENT_DIM, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.encoder = Encoder(latent_dim=latent_dim)
        self.decoder = Decoder(latent_dim=latent_dim)

    def forward(self, z):
        return self.decoder(z)

    def training_step(self, batch, batch_idx):
        x, _ = batch
        z_mean, z_log_var, z = self.encoder(x)
        x_rec = self.decoder(z)
        loss, rec_loss, kl_loss = vae_loss(x, x_rec, z_mean, z_log_var)
        self.log_dict(
            {"loss": loss, "rec_loss": rec_loss, "kl_loss": kl_loss}, prog_bar=True
        )
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
_ = L.seed_everything(SEED)

# The decoder ends with a Sigmoid, so the VAE works on images in [0, 1]
data_module_vae = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_TENSOR
)
data_module_vae.setup("fit")
monitor_images = torch.stack([data_module_vae.ds_train[i][0] for i in range(8)])

vae_model = VAE(latent_dim=VAE_LATENT_DIM)


def vae_monitor_samples(model, n=16):
    z = torch.randn(n, VAE_LATENT_DIM, device=model.device)
    return model(z)


def vae_monitor_reconstructions(model):
    x = monitor_images.to(model.device)
    _, _, z = model.encoder(x)
    return torch.cat([x, model.decoder(z)], dim=0)


fit_model(
    "VAE",
    vae_model,
    data_module_vae,
    max_epochs=VAE_EPOCHS,
    callbacks=[
        SampleGridMonitor(vae_monitor_samples, tag="vae/samples"),
        SampleGridMonitor(
            vae_monitor_reconstructions,
            tag="vae/reconstructions",
            nrow=len(monitor_images),
        ),
    ],
)

### Reconstructions vs. Samples

**Question (before running)**: The next cell shows 8 real digits, their reconstructions and 8 samples decoded from random $z \sim \mathcal{N}(0, I)$. Which will look better: the reconstructions or the samples? Will either be as sharp as the originals?

In [ ]:
_ = vae_model.eval()
x = monitor_images.to(vae_model.device)
with torch.no_grad():
    _, _, z = vae_model.encoder(x)
    x_rec = vae_model.decoder(z)
    x_sampled = vae_model.decoder(torch.randn(len(x), VAE_LATENT_DIM, device=x.device))

fig, ax = show_samples(
    torch.cat([x, x_rec, x_sampled]),
    nrow=len(x),
    title="Rows: originals / reconstructions / random samples",
)
plt.show()

<details>
<summary>Click to reveal answer</summary>

**Reconstructions look better than samples, and both are blurrier than the originals.**

- *Reconstructions* start from the encoder's $z$ for a real image, so they land in a region of latent space the decoder knows well.
- *Samples* use a random $z$ from the prior. Some land between digit clusters, which gives ambiguous "in-between" digits.
- *Blur*: with only 2 latent dimensions the decoder cannot store fine details, and the pixel-wise loss rewards predicting the *average* of all plausible images and averages are blurry. This blurriness is a classic VAE weakness.

</details>

### Latent Space Exploration

With a 2-D latent space we can visualise the entire learned manifold by decoding points on a regular grid.

In [ ]:
@torch.no_grad()
def get_latent_space_samples(
    decoder, num_points_per_dim=20, latent_range=3.0, device="cpu"
):
    grid = torch.linspace(-latent_range, latent_range, num_points_per_dim)
    xs, ys = torch.meshgrid(grid, grid, indexing="xy")
    z = torch.stack([xs.flatten(), ys.flatten()], dim=1).to(device)
    return decoder(z).cpu()


_ = vae_model.eval()
samples = get_latent_space_samples(
    vae_model.decoder, num_points_per_dim=20, device=vae_model.device
)
fig, ax = show_samples(samples, nrow=20, title="VAE latent space (2-D grid)")
plt.show()

Now the opposite direction: we **encode** 5000 training images and plot where their means $\mu$ land in the 2-D latent space, coloured by the digit label (which the VAE never saw).

In [ ]:
loader = DataLoader(data_module_vae.ds_train, batch_size=1000)
z_means, labels = [], []
with torch.no_grad():
    for i, (x, y) in enumerate(loader):
        if i == 5:
            break
        z_mean, _, _ = vae_model.encoder(x.to(vae_model.device))
        z_means.append(z_mean.cpu())
        labels.append(y)
z_means, labels = torch.cat(z_means).numpy(), torch.cat(labels).numpy()

fig, ax = plt.subplots(figsize=(7, 6))
_ = sns.scatterplot(
    x=z_means[:, 0],
    y=z_means[:, 1],
    hue=labels.astype(str),
    hue_order=[str(d) for d in range(10)],
    palette="tab10",
    s=8,
    linewidth=0,
    ax=ax,
)
_ = ax.set_xlabel("$z_1$")
_ = ax.set_ylabel("$z_2$")
_ = ax.set_title("Encoded training images (mean of q(z|x))")
_ = ax.legend(title="digit", markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

**Question**: Without using any labels during training, the latent space organises digit identity spatially. Why does this happen? And why are the clusters packed around the origin instead of spread far apart?

<details>
<summary>Click to reveal answer</summary>

**The reconstruction loss forces similar images to get similar codes.** With only 2 numbers per image, the cheapest way to reconstruct well is to give images that look alike (same digit, similar style) nearby codes — so that a small change in $z$ means a small change in the image. Digit identity is the biggest source of visual variation in MNIST, so it becomes the dominant structure.

**The KL term pulls everything towards the prior** $\mathcal{N}(0, I)$. It penalises means far from 0 and very small variances, so the clusters stay packed around the origin and overlap slightly. That is what makes random sampling from $\mathcal{N}(0, I)$ (and the smooth grid above) work.

</details>

## 3) Diffusion Models (DDPM)

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

Diffusion Models (DMs) learn to reverse a gradual noising process:

**Forward process** (fixed): $q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) I)$

**Backward process** (learned): A UNet $\epsilon_\theta(x_t, t)$ predicts the noise added at step $t$.

**Training objective**: minimise the mean-squared error between the true noise $\epsilon$ and the predicted noise $\hat{\epsilon}_\theta(x_t, t)$:
$$\mathcal{L}_{\text{DM}} = \mathbb{E}_{x_0, \epsilon, t} \left[ \| \epsilon - \hat{\epsilon}_\theta(x_t, t) \|^2 \right]$$

### Visualising the Forward Process

Let's see how the scheduler progressively destroys an image. The DDPM scheduler uses $T = 1000$ steps.

**Question (before running)**: At $t = 500$, half-way through the forward process, how much of the digit do you expect to still see?

In [ ]:
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)

# Load a single image (scaled to [-1, 1], 32x32) and add noise at increasing timesteps
x0 = TO_PM1_32(ds_mnist_train[0][0]).unsqueeze(0)

steps = [0, 100, 250, 500, 750, 999]
noised_images = []
for t in steps:
    eps = torch.randn_like(x0)
    x_t = noise_scheduler.add_noise(x0, eps, torch.tensor([t]))
    noised_images.append(denorm(x_t.squeeze(0)))

fig, ax = plot_collage(
    noised_images, captions=[f"t={t}" for t in steps], nrows=1, ncols=6
)
plt.show()

The forward process mixes signal and noise: $x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \epsilon$. Let's plot both coefficients over $t$.

In [ ]:
alphas_cumprod = noise_scheduler.alphas_cumprod
t_axis = torch.arange(len(alphas_cumprod))

fig, ax = plt.subplots(figsize=(7, 3.5))
_ = sns.lineplot(
    x=t_axis.numpy(),
    y=alphas_cumprod.sqrt().numpy(),
    label=r"signal $\sqrt{\bar{\alpha}_t}$",
    ax=ax,
)
_ = sns.lineplot(
    x=t_axis.numpy(),
    y=(1 - alphas_cumprod).sqrt().numpy(),
    label=r"noise $\sqrt{1 - \bar{\alpha}_t}$",
    ax=ax,
)
_ = ax.axvline(500, color="grey", linestyle="--", linewidth=1)
_ = ax.set_xlabel("timestep t")
_ = ax.set_ylabel("coefficient")
_ = ax.set_title("DDPM linear noise schedule")
plt.show()

<details>
<summary>Click to reveal answer</summary>

**Almost nothing.** At $t = 500$ the signal coefficient is only about 0.28 while the noise coefficient is about 0.96 — you may still guess a blob, but the digit is essentially gone. With the linear schedule most of the image is destroyed in the first half; the second half goes from "mostly noise" to "pure noise".

This matters for training: the network sees many timesteps where the input is nearly pure noise. Later schedules (e.g. the *cosine* schedule) and flow matching distribute the destruction more evenly.

</details>

### UNet Denoiser

We use a small `UNet2DModel` from `diffusers` as our denoising network $\epsilon_\theta$. It is built by `make_unet()` in the shared utilities.

In [ ]:
print(f"UNet parameters: {sum(p.numel() for p in make_unet().parameters()):,}")

### Task 3.1: DDPM Loss

Implement `ddpm_loss(denoiser, scheduler, x_zero)`:

1. Sample random **timesteps** $t$ uniformly from $[0, T)$ — one per image (`scheduler.config.num_train_timesteps` is $T$).
2. Sample **Gaussian noise** $\epsilon$ with the same shape as `x_zero`.
3. **Add noise**: `x_t = scheduler.add_noise(x_zero, eps, timesteps)`.
4. **Predict the noise**: `denoiser(x_t, timesteps).sample`.
5. Return the **MSE** between the predicted and true noise.

Create new tensors on the same device as `x_zero`.

In [ ]:
# YOUR CODE HERE
def ddpm_loss(denoiser, scheduler, x_zero):
    """Noise-prediction loss of DDPM for a batch of clean images x_zero."""
    num_steps = scheduler.config.num_train_timesteps
    timesteps = torch.randint(0, num_steps, (x_zero.shape[0],), device=x_zero.device)
    eps = torch.randn_like(x_zero)
    x_t = scheduler.add_noise(x_zero, eps, timesteps)
    eps_hat = denoiser(x_t, timesteps).sample
    return F.mse_loss(eps_hat, eps)

In [ ]:
from types import SimpleNamespace


def predict_zero(x_t, timesteps):
    """A useless denoiser that always predicts zero noise."""
    return SimpleNamespace(sample=torch.zeros_like(x_t))


x_test = torch.zeros(256, 1, 32, 32)
loss = ddpm_loss(predict_zero, noise_scheduler, x_test)
assert loss.ndim == 0, "The loss should be a scalar"
# Predicting zero noise costs E[eps^2] = 1
assert torch.isclose(loss, torch.tensor(1.0), atol=0.02), loss
print("OK")

### Diffusion Lightning Module

In [ ]:
class DM(L.LightningModule):
    def __init__(self, T=DDPM_T, beta_0=1e-4, beta_T=0.02, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.noise_scheduler = DDPMScheduler(
            num_train_timesteps=T, beta_start=beta_0, beta_end=beta_T
        )
        self.denoising = make_unet()

    def training_step(self, batch, batch_idx):
        x_zero, _ = batch
        loss = ddpm_loss(self.denoising, self.noise_scheduler, x_zero)
        self.log("loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

### The Reverse Process

To generate an image we start from pure noise $x_T \sim \mathcal{N}(0, I)$ and apply the trained denoiser **once per timestep**, from $t = T-1$ down to $0$. At each step the scheduler uses the predicted noise $\hat{\epsilon}_\theta(x_t, t)$ to compute a slightly less noisy $x_{t-1}$ (`scheduler.step`).

`ddpm_sample` implements this loop and can optionally return snapshots along the way.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, n=16, num_snapshots=0):
    """Generate n images in [0, 1] with the full DDPM reverse process.

    If num_snapshots > 0, also return that many intermediate states (in [0, 1]).
    """
    scheduler = model.noise_scheduler
    scheduler.set_timesteps(scheduler.config.num_train_timesteps)
    x = torch.randn(n, 1, 32, 32, device=model.device)  # x_T: pure noise

    snapshot_steps = set(
        torch.linspace(0, len(scheduler.timesteps) - 1, num_snapshots).long().tolist()
    )
    snapshots = []
    for i, t in enumerate(scheduler.timesteps):
        eps_hat = model.denoising(x, t).sample
        x = scheduler.step(eps_hat, t, x).prev_sample
        if i in snapshot_steps:
            snapshots.append(denorm(x).cpu())

    samples = denorm(x).cpu()
    return (samples, snapshots) if num_snapshots else samples

In [ ]:
_ = L.seed_everything(SEED)

data_module_dm = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_PM1_32
)
dm_model = DM()

fit_model(
    "DDPM",
    dm_model,
    data_module_dm,
    max_epochs=DDPM_EPOCHS,
    precision=UNET_PRECISION,
    callbacks=[
        SampleGridMonitor(
            ddpm_sample,
            tag="ddpm/samples",
            every_n_epochs=DDPM_MONITOR_EVERY_N_EPOCHS,
        )
    ],
)

### Sample from the Trained Diffusion Model

We generate 16 digits and keep 8 snapshots of each trajectory (from left: early → late in the reverse process). We also measure how long sampling takes.

In [ ]:
_ = dm_model.to(device).eval()  # Lightning moves the model back to CPU after fit

start = time.perf_counter()
samples, snapshots = ddpm_sample(dm_model, n=16, num_snapshots=8)
TIMINGS["DDPM"]["sample_s"] = time.perf_counter() - start
num_steps = dm_model.hparams.T
print(
    f"Sampling 16 images with {num_steps} steps took {TIMINGS['DDPM']['sample_s']:.1f}s"
)

fig, ax = show_samples(samples, nrow=4, title="DDPM samples")
plt.show()

# rows: 6 samples, columns: snapshots along the reverse process
trajectories = torch.stack(snapshots, dim=1)[:6].flatten(0, 1)
fig, ax = show_samples(trajectories, nrow=len(snapshots), title="Reverse process")
plt.show()

**Question**: Look at the reverse-process rows. At which point is it decided *which* digit will appear, and what happens in the remaining steps?

<details>
<summary>Click to reveal answer</summary>

**The coarse structure — which digit, where, how thick — is decided early; the late steps only refine details.** The early snapshots are still very noisy, but the rough shape is often already visible half-way through; the last few hundred steps mostly remove the remaining noise and sharpen strokes.

This suggests that many of the 1000 steps are "wasted" on small corrections — which is exactly what faster samplers (DDIM) and flow matching exploit.

</details>

## 4) Flow Matching

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

Diffusion models learn to remove noise step by step along a fixed, curved noising process. **Flow Matching** (FM) takes a more direct route: it learns a **velocity field** $v_\theta(x, t)$ that *transports* noise into data by following the ordinary differential equation (ODE)

$$\frac{dx_t}{dt} = v_\theta(x_t, t), \qquad t \in [0, 1].$$

We use the simplest choice of path (*rectified flow*): a **straight line** between a noise sample $x_0 \sim \mathcal{N}(0, I)$ and a data sample $x_1$:

$$x_t = (1 - t)\, x_0 + t\, x_1 \qquad\Rightarrow\qquad \frac{dx_t}{dt} = x_1 - x_0.$$

**Training objective**: regress the velocity along these lines:
$$\mathcal{L}_{\text{FM}} = \mathbb{E}_{x_0, x_1, t} \left[ \| v_\theta(x_t, t) - (x_1 - x_0) \|^2 \right]$$

**Sampling**: start from noise $x_0$ and integrate the ODE with a few **Euler steps**: $x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$.

We use **the same UNet and the same data** as for DDPM — only the training target and the sampler change.

> **Convention**: here $t = 0$ is noise and $t = 1$ is data. Stable Diffusion 3, FLUX and `diffusers`' `FlowMatchEulerDiscreteScheduler` use the reverse direction ($t = 1$ is noise) — same idea, flipped time axis.

### Task 4.1: The Straight-Line Path

Implement `interpolate(x0, x1, t)`. It returns the point on the line `x_t = (1 - t) * x0 + t * x1` and the target velocity `x1 - x0`.

`x0` and `x1` have shape `(B, C, H, W)`, while `t` has shape `(B,)` — one time per image. Reshape `t` so that it broadcasts over the image dimensions.

In [ ]:
# YOUR CODE HERE
def interpolate(x0, x1, t):
    """Point x_t on the straight line from noise x0 (t=0) to data x1 (t=1), and its velocity."""
    t = t.view(-1, 1, 1, 1)  # (B,) -> (B, 1, 1, 1) to broadcast over C, H, W
    x_t = (1 - t) * x0 + t * x1
    v_target = x1 - x0
    return x_t, v_target

In [ ]:
x0_test, x1_test = torch.randn(4, 1, 32, 32), torch.randn(4, 1, 32, 32)

x_t, v_target = interpolate(x0_test, x1_test, torch.zeros(4))
assert x_t.shape == x0_test.shape, f"Got {x_t.shape}"
assert torch.allclose(x_t, x0_test), "t=0 should give the noise x0"
assert torch.allclose(v_target, x1_test - x0_test), "The velocity points from x0 to x1"

x_t, _ = interpolate(x0_test, x1_test, torch.ones(4))
assert torch.allclose(x_t, x1_test), "t=1 should give the data x1"

# One time per image: image 0 at t=0, image 1 at t=1
x_t, _ = interpolate(x0_test[:2], x1_test[:2], torch.tensor([0.0, 1.0]))
assert torch.allclose(x_t[0], x0_test[0]) and torch.allclose(x_t[1], x1_test[1])
print("OK")

**Question (before running)**: The next cell shows a digit along the straight path from noise ($t=0$) to data ($t=1$). How visible will the digit be half-way, at $t = 0.5$? Compare with DDPM at $t = 500$.

In [ ]:
_ = torch.manual_seed(0)
x1_digit = TO_PM1_32(ds_mnist_train[0][0]).unsqueeze(0)
x0_noise = torch.randn_like(x1_digit)

ts = [0.0, 0.25, 0.5, 0.75, 1.0]
path_images = [
    denorm(interpolate(x0_noise, x1_digit, torch.tensor([t]))[0].squeeze(0)) for t in ts
]
fig, ax = plot_collage(path_images, captions=[f"t={t}" for t in ts], nrows=1, ncols=5)
plt.show()

<details>
<summary>Click to reveal answer</summary>

**The digit is clearly visible at $t = 0.5$** — signal and noise contribute equally (both coefficients are 0.5). With DDPM's linear schedule the signal coefficient at $t = 500$ is only about 0.28, so the digit is almost gone.

The straight path changes the image *at a constant rate*; there is no long stretch of "almost pure noise" as in DDPM.

</details>

### Task 4.2: Flow Matching Loss

Implement `fm_loss(velocity, x1)` for a batch of clean images `x1`. `velocity(x_t, t)` is the network: it takes images and a tensor of times of shape `(B,)` and returns the predicted velocity (same shape as `x_t`).

1. Sample noise `x0` with the same shape as `x1`.
2. Sample one time `t ~ U(0, 1)` per image (`torch.rand`).
3. Compute `x_t` and the target velocity with your `interpolate`.
4. Return the **MSE** between the predicted and the target velocity.

Compare with your `ddpm_loss` — the structure is almost identical.

In [ ]:
# YOUR CODE HERE
def fm_loss(velocity, x1):
    """Flow matching loss: regress the velocity of the straight noise -> data path."""
    x0 = torch.randn_like(x1)
    t = torch.rand(x1.shape[0], device=x1.device)
    x_t, v_target = interpolate(x0, x1, t)
    return F.mse_loss(velocity(x_t, t), v_target)

In [ ]:
_ = torch.manual_seed(0)
x1_test = torch.zeros(256, 1, 32, 32)

# A useless network that predicts zero velocity costs E[(x1 - x0)^2] = 1 for x1 = 0
loss = fm_loss(lambda x_t, t: torch.zeros_like(x_t), x1_test)
assert loss.ndim == 0, "The loss should be a scalar"
assert torch.isclose(loss, torch.tensor(1.0), atol=0.02), loss

# An "oracle" that knows the target x1 can compute the exact velocity (x1 - x_t) / (1 - t)
x1_test = torch.ones(64, 1, 32, 32)
oracle = lambda x_t, t: (x1_test - x_t) / (1 - t).view(-1, 1, 1, 1)  # noqa: E731
assert fm_loss(oracle, x1_test) < 1e-3, "Check the direction of the velocity: x1 - x0"
print("OK")

### Flow Matching Lightning Module

`velocity(x, t)` wraps the UNet. The UNet's timestep embedding was designed for integer DDPM steps in $[0, 1000)$, so we pass $t \cdot 1000$ to keep the same input range.

In [ ]:
class FlowMatching(L.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.unet = make_unet()

    def velocity(self, x, t):
        return self.unet(x, t * 1000).sample

    def training_step(self, batch, batch_idx):
        x1, _ = batch
        loss = fm_loss(self.velocity, x1)
        self.log("loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

### Task 4.3: Euler Sampler

Implement `euler_sample(velocity, x0, num_steps)`: integrate the ODE from $t = 0$ to $t = 1$ in `num_steps` equal steps of size $\Delta t = 1 / \text{num\_steps}$:

$$x \leftarrow x + \Delta t \cdot v(x, t), \qquad t = 0, \Delta t, 2\Delta t, \dots, 1 - \Delta t$$

Pass `t` to `velocity` as a tensor of shape `(B,)`. Return the final `x`.

In [ ]:
# YOUR CODE HERE
@torch.no_grad()
def euler_sample(velocity, x0, num_steps):
    """Integrate dx/dt = velocity(x, t) from t=0 (noise x0) to t=1 with Euler steps."""
    dt = 1.0 / num_steps
    x = x0
    for i in range(num_steps):
        t = torch.full((x.shape[0],), i * dt, device=x.device)
        x = x + dt * velocity(x, t)
    return x

In [ ]:
x0_test = torch.randn(4, 1, 32, 32)
target = torch.ones(4, 1, 32, 32)

# A constant velocity c moves x0 to x0 + c
x_end = euler_sample(lambda x, t: torch.full_like(x, 2.0), x0_test, num_steps=10)
assert x_end.shape == x0_test.shape, f"Got {x_end.shape}"
assert torch.allclose(x_end, x0_test + 2.0, atol=1e-5)

# The straight-line field towards a fixed target is integrated *exactly* by Euler,
# no matter how few steps we take
straight = lambda x, t: (target - x) / (1 - t).view(-1, 1, 1, 1)  # noqa: E731
for num_steps in [1, 3, 10]:
    x_end = euler_sample(straight, x0_test, num_steps)
    assert torch.allclose(x_end, target, atol=1e-4), f"num_steps={num_steps}"
print("OK")

### Training

`fm_sample` generates images from a trained model with your Euler sampler. Training uses the same data, UNet and number of epochs as DDPM.

In [ ]:
def fm_sample(model, n=16, num_steps=FM_SAMPLE_STEPS, x0=None):
    """Generate n images in [0, 1] with the Euler sampler (optionally from given noise x0)."""
    if x0 is None:
        x0 = torch.randn(n, 1, 32, 32)
    return denorm(euler_sample(model.velocity, x0.to(model.device), num_steps)).cpu()


_ = L.seed_everything(SEED)

data_module_fm = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_PM1_32
)
fm_model = FlowMatching()

fit_model(
    "FM",
    fm_model,
    data_module_fm,
    max_epochs=FM_EPOCHS,
    precision=UNET_PRECISION,
    callbacks=[SampleGridMonitor(fm_sample, tag="fm/samples")],
)

### Sample from the Flow Matching Model

We generate 16 digits with `FM_SAMPLE_STEPS` Euler steps and measure the sampling time.

In [ ]:
_ = fm_model.to(device).eval()  # Lightning moves the model back to CPU after fit

start = time.perf_counter()
samples = fm_sample(fm_model, n=16)
TIMINGS["FM"]["sample_s"] = time.perf_counter() - start
print(
    f"FM:   16 images with {FM_SAMPLE_STEPS:>4} steps took {TIMINGS['FM']['sample_s']:.1f}s"
)
if "sample_s" in TIMINGS.get("DDPM", {}):
    num_steps = TRAINED_MODELS["DDPM"].hparams.T
    print(
        f"DDPM: 16 images with {num_steps:>4} steps took {TIMINGS['DDPM']['sample_s']:.1f}s"
    )
else:
    print("(DDPM section was skipped — no comparison available)")

fig, ax = show_samples(samples, nrow=4, title=f"FM samples ({FM_SAMPLE_STEPS} steps)")
plt.show()

### How Many Steps Do We Need?

**Question (before running)**: The next cell generates images from the *same* starting noise with 1, 2, 5, 10, 20 and 50 Euler steps. With how many steps do you expect recognisable digits? What will a single step produce?

In [ ]:
_ = torch.manual_seed(0)
x0_fixed = torch.randn(6, 1, 32, 32)
step_counts = [1, 2, 5, 10, 20, 50]

# rows: 6 starting noises, columns: number of Euler steps
grid = torch.stack(
    [fm_sample(fm_model, num_steps=s, x0=x0_fixed) for s in step_counts], dim=1
).flatten(0, 1)
fig, ax = show_samples(
    grid, nrow=len(step_counts), title="Columns: 1, 2, 5, 10, 20, 50 Euler steps"
)
plt.show()

<details>
<summary>Click to reveal answer</summary>

**One step gives a blurry "average digit"; a handful of steps already give recognisable digits; beyond ~20 steps little changes.**

With a single step the sampler jumps from $x_0$ to $x_0 + v_\theta(x_0, 0)$. At $t=0$ the noise says nothing about which digit to produce, so the best the network can predict is the *average* direction over all digits — a blur.

The training paths are straight lines, but the *learned* velocity field averages over all pairs $(x_0, x_1)$ that pass through a point, which bends the actual sampling trajectories. They are still **much straighter than DDPM's**, which is why 10–20 steps instead of 1000 suffice. (Your Euler check showed the extreme case: a perfectly straight field needs a single step.)

</details>

**Question**: Is it fair to compare FM with 20 steps against DDPM with 1000 steps?

<details>
<summary>Click to reveal answer</summary>

**Not entirely.** A trained DDPM can also be sampled with fewer steps using a deterministic sampler such as **DDIM** (e.g. 20–50 steps, swap in `diffusers.DDIMScheduler`), at some cost in quality. The real advantage of flow matching is that its paths are **straighter by construction**, so a simple Euler solver works with few steps, and the training objective is simpler (no noise schedule to design).

This is why recent large text-to-image models such as **Stable Diffusion 3** and **FLUX** are trained with flow matching (rectified flow).

</details>

## 5) Bonus: Autoregressive Generation (VQ-VAE + Transformer)

> **Bonus section** — skip to **Comparison** if you are short on time.

*This section only needs the cells up to and including **0) Prepare Dataset** — you can skip the other model sections.*

Large language models generate text **one token at a time**: $p(x) = \prod_i p(x_i \mid x_{<i})$. We can do exactly the same with images — if we first turn an image into a sequence of tokens. This takes two stages:

1. **Tokenizer (VQ-VAE)**: an encoder maps a 28x28 image to a 7x7 grid of vectors. Each vector is replaced by its nearest entry in a learned **codebook** of $K$ vectors — so every image becomes 49 integers (token ids) in $[0, K)$. A decoder maps the (quantized) grid back to an image.
2. **Prior (Transformer)**: a small GPT-style Transformer learns to predict the next token id given the previous ones — next-token prediction, just like an LLM. We prepend a **class token** ("draw a 7") so that we can choose the digit.

This is the recipe behind DALL·E 1, Parti and LlamaGen (with much larger tokenizers and Transformers).

Compared with the VAE from Section 2: the latent is no longer a continuous Gaussian vector but a **grid of discrete codes**.

### Task 5.1: Vector Quantization

Implement `quantize(z_e, codebook)`:

- `z_e` has shape `(B, D, H, W)`: one $D$-dimensional vector per grid cell.
- `codebook` has shape `(K, D)`.

Steps:
1. Reshape `z_e` into a list of vectors of shape `(B*H*W, D)` (move `D` to the last dimension first).
2. For every vector find the index of the **nearest codebook entry** (Euclidean distance; `torch.cdist` + `argmin`).
3. Look up the codebook vectors and reshape back to `(B, D, H, W)` → `z_q`.
4. **Straight-through estimator**: `argmin` has no gradient. Return `z_e + (z_q - z_e).detach()`, which has the *value* of `z_q` but passes the gradient unchanged to `z_e`.

Return `(z_q, indices)` with `indices` of shape `(B, H, W)`.

In [ ]:
# YOUR CODE HERE
def quantize(z_e, codebook):
    """Replace every vector in z_e (B, D, H, W) by its nearest codebook entry (K, D)."""
    B, D, H, W = z_e.shape
    z_flat = z_e.permute(0, 2, 3, 1).reshape(-1, D)  # (B*H*W, D)
    indices = torch.cdist(z_flat, codebook).argmin(dim=1)  # nearest code per vector
    z_q = codebook[indices].view(B, H, W, D).permute(0, 3, 1, 2)
    z_q = z_e + (z_q - z_e).detach()  # straight-through estimator
    return z_q, indices.view(B, H, W)

In [ ]:
codebook_test = torch.tensor([[0.0, 0.0], [1.0, 1.0], [-1.0, 2.0]])  # K=3, D=2
vectors = torch.tensor([[0.9, 1.2], [-0.8, 1.9], [0.1, -0.1], [1.0, 1.0]])  # 4 cells
z_e_test = vectors.T.reshape(1, 2, 2, 2).clone().requires_grad_(True)  # (B, D, H, W)

z_q, indices = quantize(z_e_test, codebook_test)
assert indices.shape == (1, 2, 2), f"Got {indices.shape}"
assert indices.flatten().tolist() == [1, 2, 0, 1], indices
assert z_q.shape == z_e_test.shape
assert torch.allclose(z_q.detach(), codebook_test[indices].permute(0, 3, 1, 2))

z_q.sum().backward()
assert torch.allclose(z_e_test.grad, torch.ones_like(z_e_test)), (
    "Use the straight-through estimator: z_e + (z_q - z_e).detach()"
)
print("OK")

### VQ-VAE Lightning Module

Besides the reconstruction loss, a VQ-VAE needs two extra terms, because the quantization step itself has no gradient:

- **Codebook loss** $\|\text{sg}[z_e] - z_q\|^2$ moves the codebook vectors towards the encoder outputs (sg = stop-gradient, `.detach()`).
- **Commitment loss** $\beta \|z_e - \text{sg}[z_q]\|^2$ keeps the encoder outputs close to their codes.

A common failure is **codebook collapse**: only a few codes are ever chosen, the rest are "dead". We therefore re-initialise unused codes with random encoder outputs every 100 steps and log the fraction of codes in use (`codebook_usage`).

In [ ]:
class VQVAE(L.LightningModule):
    def __init__(
        self, num_codes=VQ_NUM_CODES, code_dim=VQ_CODE_DIM, beta=0.25, lr=1e-3
    ):
        super().__init__()
        self.save_hyperparameters()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=4, stride=2, padding=1),  # 28 -> 14
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 14 -> 7
            nn.ReLU(),
            nn.Conv2d(64, code_dim, kernel_size=1),
        )
        self.codebook = nn.Embedding(num_codes, code_dim)
        self.decoder = nn.Sequential(
            nn.Conv2d(code_dim, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),  # 7 -> 14
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=4, stride=2, padding=1),  # 14 -> 28
            nn.Sigmoid(),
        )
        self.register_buffer("code_counts", torch.zeros(num_codes))

    def lookup(self, indices):
        """Token ids (B, H, W) -> codebook vectors (B, D, H, W)."""
        return self.codebook(indices).permute(0, 3, 1, 2)

    def encode_tokens(self, x):
        """Images (B, 1, 28, 28) -> token ids (B, 49)."""
        _, indices = quantize(self.encoder(x), self.codebook.weight)
        return indices.flatten(1)

    def decode_tokens(self, tokens):
        """Token ids (B, 49) -> images (B, 1, 28, 28) in [0, 1]."""
        return self.decoder(self.lookup(tokens.view(-1, 7, 7)))

    def training_step(self, batch, batch_idx):
        x, _ = batch
        z_e = self.encoder(x)
        z_q, indices = quantize(z_e, self.codebook.weight)
        x_rec = self.decoder(z_q)

        rec_loss = F.binary_cross_entropy(x_rec, x)
        z_q_hard = self.lookup(indices)  # without the straight-through trick
        codebook_loss = F.mse_loss(z_q_hard, z_e.detach())
        commitment_loss = F.mse_loss(z_e, z_q_hard.detach())
        loss = rec_loss + codebook_loss + self.hparams.beta * commitment_loss

        self.code_counts += torch.bincount(
            indices.flatten(), minlength=self.hparams.num_codes
        )
        if (batch_idx + 1) % 100 == 0:
            self.reset_dead_codes(z_e.detach())

        self.log_dict({"loss": loss, "rec_loss": rec_loss}, prog_bar=True)
        return loss

    @torch.no_grad()
    def reset_dead_codes(self, z_e):
        dead = self.code_counts == 0
        self.log("codebook_usage", 1 - dead.float().mean(), prog_bar=True)
        if dead.any():
            z_flat = z_e.permute(0, 2, 3, 1).reshape(-1, self.hparams.code_dim)
            random_rows = torch.randint(
                len(z_flat), (int(dead.sum()),), device=z_e.device
            )
            self.codebook.weight[dead] = z_flat[random_rows]
        self.code_counts.zero_()

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
_ = L.seed_everything(SEED)

data_module_vq = MNISTDataModule(
    data_dir=DATA_PATH.joinpath("mnist"), transform_fn=TO_TENSOR
)
data_module_vq.setup("fit")
vq_monitor_images = torch.stack([data_module_vq.ds_train[i][0] for i in range(8)])

vqvae_model = VQVAE()


def vq_monitor_reconstructions(model):
    x = vq_monitor_images.to(model.device)
    return torch.cat([x, model.decode_tokens(model.encode_tokens(x))], dim=0)


fit_model(
    "VQ-VAE",
    vqvae_model,
    data_module_vq,
    max_epochs=VQ_EPOCHS,
    callbacks=[
        SampleGridMonitor(
            vq_monitor_reconstructions, tag="vqvae/reconstructions", nrow=8
        )
    ],
)

### Turning MNIST into Token Sequences

We encode the whole training set once. Every image becomes a sequence of 49 token ids.

In [ ]:
_ = vqvae_model.to(device).eval()  # Lightning moves the model back to CPU after fit

token_list, label_list = [], []
with torch.no_grad():
    for x, y in DataLoader(data_module_vq.ds_train, batch_size=1000):
        token_list.append(vqvae_model.encode_tokens(x.to(device)).cpu())
        label_list.append(y)
tokens, token_labels = torch.cat(token_list), torch.cat(label_list)
print(
    f"Tokens: {tuple(tokens.shape)}, {tokens.unique().numel()} of {VQ_NUM_CODES} codes in use"
)
print("Token ids of the first image (7x7):")
print(tokens[0].view(7, 7))

with torch.no_grad():
    x_rec = vqvae_model.decode_tokens(tokens[:8].to(device)).cpu()
fig, ax = show_samples(
    torch.cat([vq_monitor_images, x_rec]),
    nrow=8,
    title="Originals / decoded from 49 tokens",
)
plt.show()

**Question**: How strongly does the tokenizer compress an image? Compare the number of bits of a 28x28 8-bit image with 49 tokens from a codebook of 64 entries.
<details>
<summary>Click to reveal answer</summary>

- Image: $28 \cdot 28 \cdot 8 = 6272$ bits.
- Tokens: $49 \cdot \log_2 64 = 49 \cdot 6 = 294$ bits.

**About 21x compression** — and the decoded digits still look very similar. This is the point of the tokenizer: the Transformer only has to model 49 positions instead of 784 pixels, and each token carries meaningful local structure (stroke pieces) instead of single pixel values. The price: whatever the tokenizer cannot represent, the generator cannot produce either.

</details>

### The Transformer Prior

The sequence for an image of class $c$ is `[<class c>, t_0, t_1, ..., t_48]`. At every position the Transformer outputs a distribution over the $K$ codes for the **next** token. Training is plain **next-token prediction** with cross-entropy.

The key ingredient is the **causal mask**: position $i$ may only attend to positions $\le i$. Here is the mask for a sequence of length 8 (`-inf` entries are blocked):

In [ ]:
mask = nn.Transformer.generate_square_subsequent_mask(8)
fig, ax = plt.subplots(figsize=(3.5, 3.5))
_ = ax.imshow(torch.isinf(mask).numpy(), cmap="Greys")
_ = ax.set_xlabel("attends to position")
_ = ax.set_ylabel("query position")
_ = ax.set_title("Causal mask (black = blocked)")
plt.show()

**Question**: Why must the mask be triangular? What would happen during training without it?
<details>
<summary>Click to reveal answer</summary>

**Without the mask the model could simply look at the token it is supposed to predict.** During training the whole sequence is fed in at once, so position $i$ could attend to $t_{i}$ (its own target) and copy it — the training loss would go to ~0 while the model learns nothing useful. At generation time future tokens do not exist yet, so such a model would be useless. The mask makes training match generation: each prediction only uses the past.

</details>

In [ ]:
class TokenTransformer(L.LightningModule):
    def __init__(
        self,
        num_codes=VQ_NUM_CODES,
        num_classes=10,
        seq_len=49,
        d_model=128,
        num_layers=4,
        num_heads=4,
        lr=1e-3,
    ):
        super().__init__()
        self.save_hyperparameters()
        # Vocabulary: K image codes + one start token per class
        self.token_emb = nn.Embedding(num_codes + num_classes, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model,
            num_heads,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            layer, num_layers, enable_nested_tensor=False
        )
        self.head = nn.Linear(d_model, num_codes)

    def start_tokens(self, labels):
        return labels + self.hparams.num_codes

    def forward(self, seq):
        """Token ids (B, L) -> logits for the next token at every position (B, L, K)."""
        length = seq.shape[1]
        positions = torch.arange(length, device=seq.device)
        h = self.token_emb(seq) + self.pos_emb(positions)
        mask = nn.Transformer.generate_square_subsequent_mask(length, device=seq.device)
        h = self.transformer(h, mask=mask, is_causal=True)
        return self.head(h)

    def training_step(self, batch, batch_idx):
        tokens, labels = batch  # (B, 49), (B,)
        # Input: [<class>, t_0, ..., t_47] -> targets: [t_0, ..., t_48]
        seq = torch.cat([self.start_tokens(labels)[:, None], tokens[:, :-1]], dim=1)
        logits = self(seq)
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tokens.reshape(-1))
        self.log("loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.hparams.lr)

### Task 5.2: Autoregressive Sampling

Implement `sample_tokens(model, start_tokens, num_tokens, temperature)`. Starting from the sequence `start_tokens[:, None]` (shape `(B, 1)`), repeat `num_tokens` times:

1. Run the model on the current sequence and take the logits of the **last** position: `model(seq)[:, -1]` → `(B, K)`.
2. Divide the logits by `temperature` and convert them to probabilities (`softmax`).
3. Sample one token per sequence with `torch.multinomial(probs, num_samples=1)`.
4. Append it to the sequence.

Return the generated tokens **without** the start token, shape `(B, num_tokens)`.

In [ ]:
# YOUR CODE HERE
@torch.no_grad()
def sample_tokens(model, start_tokens, num_tokens, temperature=1.0):
    """Generate num_tokens tokens one at a time, each conditioned on all previous ones."""
    seq = start_tokens[:, None]  # (B, 1)
    for _ in range(num_tokens):
        logits = model(seq)[:, -1] / temperature  # (B, K): logits for the next token
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # (B, 1)
        seq = torch.cat([seq, next_token], dim=1)
    return seq[:, 1:]

In [ ]:
def counting_model(seq):
    """A fake model that (almost surely) predicts 'previous token + 1' (mod 10)."""
    return F.one_hot((seq + 1) % 10, num_classes=10).float() * 100


out = sample_tokens(counting_model, torch.tensor([0, 5]), num_tokens=12)
assert out.shape == (2, 12), f"Got {tuple(out.shape)}"
assert out[0].tolist() == [1, 2, 3, 4, 5, 6, 7, 8, 9, 0, 1, 2], out[0]
assert out[1].tolist() == [6, 7, 8, 9, 0, 1, 2, 3, 4, 5, 6, 7], (
    "Each new token must be fed back into the model"
)
print("OK")

In [ ]:
@torch.no_grad()
def ar_sample(prior, tokenizer, labels, temperature=1.0):
    """Generate one image per label in [0, 1]: sample 49 tokens, then decode them."""
    start = prior.start_tokens(labels.to(prior.device))
    tokens = sample_tokens(prior, start, num_tokens=49, temperature=temperature)
    return tokenizer.decode_tokens(tokens.to(tokenizer.device)).cpu()


_ = L.seed_everything(SEED)

token_loader = DataLoader(
    TensorDataset(tokens, token_labels), batch_size=BATCH_SIZE, shuffle=True
)
ar_model = TokenTransformer()

fit_model(
    "AR",
    ar_model,
    token_loader,
    max_epochs=AR_EPOCHS,
    callbacks=[
        SampleGridMonitor(
            lambda m: ar_sample(m, vqvae_model, torch.arange(10)),
            tag="ar/samples",
            nrow=10,
        )
    ],
)

### Draw a Digit on Request

Each row uses the class token of one digit (0–9); each column is a new sample.

In [ ]:
_ = ar_model.to(device).eval()  # Lightning moves the model back to CPU after fit
_ = vqvae_model.to(device).eval()

labels = torch.arange(10).repeat_interleave(8)  # 8 samples per digit
start = time.perf_counter()
samples = ar_sample(ar_model, vqvae_model, labels)
TIMINGS["AR"]["sample_s"] = time.perf_counter() - start
print(
    f"Sampling {len(labels)} images with 49 steps took {TIMINGS['AR']['sample_s']:.1f}s"
)

fig, ax = show_samples(samples, nrow=8, title="Rows: class token 0-9")
plt.show()

**Question (before running)**: The next cell samples the digit 4 with temperatures 0.3, 1.0 and 2.0 (one row each). What do you expect to change?

In [ ]:
_ = torch.manual_seed(SEED)
temperatures = [0.3, 1.0, 2.0]
rows = [
    ar_sample(ar_model, vqvae_model, torch.full((8,), 4), temperature=t)
    for t in temperatures
]
fig, ax = show_samples(torch.cat(rows), nrow=8, title="Temperature 0.3 / 1.0 / 2.0")
plt.show()

<details>
<summary>Click to reveal answer</summary>

**Low temperature → safe but similar samples; high temperature → diverse but broken samples.**

Dividing the logits by $T < 1$ sharpens the distribution, so the model almost always picks its most likely token: clean, "typical" 4s that look alike. $T > 1$ flattens the distribution, so unlikely tokens are picked more often: more variety, but also strokes in the wrong places — and an early bad token cannot be corrected later, because every following token builds on it.

This is exactly the `temperature` setting of LLM APIs.

</details>

**Question**: The Transformer works exactly like a (tiny) language model. What is different — and harder — about generating images this way compared with text?
<details>
<summary>Click to reveal answer</summary>

- **The order is arbitrary.** Text has a natural left-to-right order; for images we *chose* raster order (row by row). The token directly *above* the current one is 7 positions back, the one below is in the future and cannot be used.
- **Sequences are long.** 49 tokens is tiny; a 256x256 image with a 16x16 token grid already needs 256 sequential steps, and every step is a full forward pass.
- **The tokenizer limits quality.** Blurry or lossy tokens give blurry samples, no matter how good the Transformer is (this is why VQGAN adds perceptual and adversarial losses to the tokenizer).
- **No going back.** A mistake in an early token cannot be fixed later, while diffusion and flow models refine the *whole* image at every step.

Recent models address these points with e.g. predicting many tokens in parallel (MaskGIT) or coarse-to-fine "next-scale" prediction (VAR).

</details>

## Comparison

Run this section after any subset of the model sections — it uses every model in `TRAINED_MODELS` and lists the ones you skipped. All models are sampled with the same seed, and the sampling time for 16 images is measured.

In [ ]:
import pandas as pd

COMPARE_N = 16

# How to draw n samples in [0, 1] from each (trained) model
sample_fns = {
    "GAN": lambda m, n: denorm(m(torch.randn(n, m.latent_dim, device=m.device))),
    "VAE": lambda m, n: m(torch.randn(n, VAE_LATENT_DIM, device=m.device)),
    "DDPM": lambda m, n: ddpm_sample(m, n),
    "FM": lambda m, n: fm_sample(m, n),
    "AR": lambda m, n: ar_sample(m, TRAINED_MODELS["VQ-VAE"], torch.arange(n) % 10),
}
steps = {"GAN": 1, "VAE": 1, "DDPM": DDPM_T, "FM": FM_SAMPLE_STEPS, "AR": 49}

available = [name for name in sample_fns if name in TRAINED_MODELS]
skipped = [name for name in sample_fns if name not in TRAINED_MODELS]
print(f"Comparing: {available}" + (f" | skipped: {skipped}" if skipped else ""))
for model in TRAINED_MODELS.values():
    _ = model.to(device).eval()

results, grids = [], {}
for name in available:
    _ = torch.manual_seed(SEED)
    start = time.perf_counter()
    with torch.no_grad():
        images = sample_fns[name](TRAINED_MODELS[name], COMPARE_N).float().cpu()
    sample_s = time.perf_counter() - start
    grids[name] = F.interpolate(images, size=32)  # 28x28 models -> 32x32
    train_s = TIMINGS[name]["train_s"]
    if name == "AR":  # the AR model also needs its tokenizer
        train_s += TIMINGS["VQ-VAE"]["train_s"]
    results.append(
        {
            "model": name,
            "training time (s)": round(train_s),
            "network evaluations per sample": steps[name],
            f"sampling {COMPARE_N} images (s)": round(sample_s, 2),
        }
    )

fig, axes = plt.subplots(
    1, len(available), figsize=(3 * len(available), 3.4), squeeze=False
)
for ax, name in zip(axes[0], available, strict=True):
    grid = make_grid(grids[name], nrow=4, padding=2, pad_value=1.0)
    _ = ax.imshow(grid.permute(1, 2, 0).numpy())
    _ = ax.set_title(name)
    _ = ax.axis("off")
plt.show()

pd.DataFrame(results).set_index("model")

### Properties at a Glance

| | GAN | VAE | DDPM | FM | AR (VQ-VAE + Transformer) |
|---|---|---|---|---|---|
| Training objective | adversarial (min-max) | ELBO (reconstruction + KL) | noise prediction (MSE) | velocity regression (MSE) | next-token prediction (cross-entropy) |
| Training stability | fragile | stable | stable | stable | stable (two stages) |
| Sampling | 1 forward pass | 1 forward pass | ~1000 steps (fewer with DDIM) | ~10–50 ODE steps | 1 step per token |
| Sample quality | sharp, risk of mode collapse | blurry, full coverage | sharp, diverse | sharp, diverse | limited by the tokenizer |
| Useful latent space | yes ($z$) | yes ($z$, smooth) | no (noise $x_T$) | no (noise $x_0$) | discrete tokens |
| Used today in | super-resolution, fast generators, as a loss (VQGAN) | tokenizer / latent space of Stable Diffusion | many image & video models | Stable Diffusion 3, FLUX | multimodal LLMs, LlamaGen |

## Wrap-up Questions

1. How does the quality of the samples compare between the models you trained? (See the comparison above.)
2. What happens to GAN training if the discriminator becomes too strong early on?
3. How does the sampling speed of DDPM, FM (and AR) compare to GAN/VAE, and what is the trade-off?
4. DDPM and FM use the same UNet and the same data. What exactly is different between them?

<details>
<summary>Click to reveal answer</summary>

1. **Diversity vs. sharpness.** The GAN typically gives sharp digits but may miss some digits (mode collapse). The VAE covers all digits but is blurry. DDPM and FM give sharp *and* diverse samples — but they are the most expensive to train, and with only a few epochs some samples are still malformed. The AR model is about as sharp as its tokenizer allows, and can be told which digit to draw.
2. **The generator stops learning.** If $D$ rejects every fake with high confidence, the generator receives tiny or uninformative gradients. Our generator loss (target "real" for fakes — the *non-saturating* loss) and the label smoothing in the discriminator loss both mitigate this; other fixes are a lower discriminator learning rate or Wasserstein-style losses.
3. **GAN and VAE generate in a single forward pass; DDPM needs 1000, FM about 20, AR one per token (49)** — see the table. Diffusion, flow and autoregressive models are stable to train and cover the data distribution well, but pay for it with many network evaluations at sampling time.
4. **The target, the path and the sampler.**
   - DDPM predicts the *noise* $\epsilon$; FM predicts the *velocity* $x_1 - x_0$.
   - DDPM's forward process follows a curved schedule ($\sqrt{\bar\alpha_t}$, $\sqrt{1 - \bar\alpha_t}$); FM uses a straight line.
   - DDPM samples with a stochastic 1000-step chain; FM integrates a deterministic ODE with a few Euler steps.

</details>